# 劳动力轮班调度问题

**类别:** 调度

来源: [https://www.hexaly.com/templates/workforce-shift-scheduling-problem](https://www.hexaly.com/templates/workforce-shift-scheduling-problem)

## 问题

在劳动力轮班调度问题中,我们考虑固定数量的员工和一组待调度的任务(称为活动)。每个活动具有指定的持续时长,并且必须在定义的时间窗口内执行。同样,每位员工都有一个限定的工作可用性窗口。除此之外,还需满足每位员工每天最多可工作的轮班数,以及相邻轮班之间所需的最小休息时间等约束。问题的主要目标是最小化人手不足——即所有活动中未分配的员工总数——以及最小化所有员工的总工作时间。

### 学到的建模技巧

- 定义[多目标模型](https://www.hexaly.com/docs/last/modelingprinciples/multiobjectiveresolution.html)并确定每个目标的优先级
- 了解 Hexaly Optimizer 的建模风格:[区分决策变量与中间表达式](https://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-decision-variables-from-intermediate-variables)

## 数据

劳动力轮班调度问题的数据文件格式如下:

- 第一行:员工数量
- 第二行:活动数量
- 第三行:规划周期(以天为单位)
- 第四行:时间增量(以小时为单位)
- 后续行:

- 每个活动:活动的时间窗口(以小时为单位的时间区间)
- 每个员工:其可用性时间窗口(以小时为单位的时间区间)

## 模型

用于劳动力轮班调度问题的 Hexaly 模型使用布尔决策变量。每个变量表示某个员工是否被分配到特定的轮班。对于每个活动,模型生成一组可能的轮班。每个轮班代表员工可执行该活动的时间区间。我们通过使用固定时间增量在规划周期内离散化可能起始时间来生成这些轮班。

为防止排班冲突,模型强制施加不重叠约束。如果两个轮班在时间上重叠,则同一员工不能同时承担这两个轮班。该规则适用于所有不相容的轮班对。或者,模型也可以在离散化的时间线上施加该约束,即确保任意员工在任意时间槽(例如每 15 分钟或每小时)中都不会同时进行多个轮班。该方法避免了枚举所有冲突轮班对,并在轮班数量较多时可减少约束数量。两种方法的选择取决于轮班数量和总时间步数。

模型还要求同一员工的任意两个轮班之间至少休息一小时。这一约束与不重叠规则类似。我们将违反该休息时间的轮班对标记为不相容。为实现这一点,我们在检查冲突时将每个轮班的结束时间加上 1 小时。

每位员工每天必须工作恰好两个轮班。模型通过将员工的决策变量求和并令总和等于 2 来强制实施该约束。

为保证充分覆盖,模型将规划周期划分为相等的时间区间。对于每个区间和每个活动,我们检查哪些轮班与该区间重叠,然后统计这些轮班所分配的员工数。若数量不足,则记录人手不足。我们对所有活动与区间的人手不足进行求和,并将其最小化。

为减少不必要的工作量,我们最小化所有员工的总工作时间。该目标有助于促进更公平的调度,并限制过度劳动。两个目标按[字典序进行优化](https://www.hexaly.com/docs/last/modelingprinciples/multiobjectiveresolution.html):最小化人手不足是首要优先级,平衡工作量则其次。

## Python 实现

In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

# Activity class represents a work item with its scheduling constraints
#   - id: Unique identifier for the task
#   - min_start: minimum possible start time (in seconds)
#   - max_end: maximum possible end time (in seconds)
#   - duration: Duration of the task (in seconds)
class Activity:
    def __init__(self, id, min_start, max_end, duration):
        self.id = id #int
        self.min_start = min_start #int
        self.max_end = max_end #int
        self.duration = duration #int

# Agent class represents an employee with their availability window
#   - id: Unique identifier for the agent
#   - availability_start: Start of availability period (in seconds)
#   - availability_end: End of availability period (in seconds)
class Agent:
    def __init__(self, id, av_start, av_end):
        self.id = id #int
        self.availability_start = av_start #int
        self.availability_end = av_end #int

# Shift class represents a work item with its scheduling constraints
#   - id: Unique identifier for the shift
#   - activity_id: ID of the activity performed
#   - shift_start: Start time of the shift (in seconds)
#   - shift_end: End time of the shift (in seconds)
class Shift:
    def __init__(self, id, activity_id, shift_start, shift_end):
        self.id = id #int
        self.activity_id = activity_id #int
        self.shift_start = shift_start #int
        self.shift_end = shift_end #int


#  Generates all possible shifts for activities with given time increment
#  @param activities Array of Activity objects
#  @param shift_increment Time increment in seconds
#  @return Array of generated Shift objects
def generateShifts(activities, shift_increment):
    i = 0
    shifts = []
    for a in activities:
        current_time = a.min_start
        while current_time + a.duration <= a.max_end:
            shifts.append(Shift(i, a.id, current_time, current_time + a.duration))
            i += 1
            current_time += shift_increment
    return shifts


# Checks if two shifts are compatible
#   @param slot1 First shift
#   @param slot2 Second shift
#   @return true if shifts overlap, false otherwise
def incompatibleShifts(shift1, shift2):
    startShift1 = shift1.shift_start
    endShift1 = shift1.shift_end
    startShift2 = shift2.shift_start
    endShift2 = shift2.shift_end
    return not (startShift1 >= endShift2 or endShift1 <= startShift2)


# Checks if there is at least a one-hour break between two shifts
#   @param slot1 First shift
#   @param slot2 Second shift
#   @return true if there is not enough break, false otherwise
def notEnoughBreak(shift1, shift2):
    startShift1 = shift1.shift_start
    endShift1 = shift1.shift_end
    startShift2 = shift2.shift_start
    endShift2 = shift2.shift_end
    return not (startShift1 >= endShift2 + 3600 or endShift1 + 3600 <= startShift2)


# Checks if two time intervals overlap
#   @param interval1Start Start of first interval
#   @param interval1End End of first interval
#   @param interval2Start Start of second interval
#   @param interval2End End of second interval
#   @return true if intervals overlap, false otherwise
def overlappingIntervals(int1_start, int1_end, int2_start, int2_end):
    return not (int1_start >= int2_end or int1_end <= int2_start)


if len(sys.argv) < 2:
    print("Usage: python workforce_scheduling_shifts.py inputFile [outputFile] [timeLimit]")
    sys.exit(1)

# Filters out comments from a text file
#   @param filename Path to input file
def read_elem(filename):
    with open(filename) as f:
        lines = f.readlines()
        result = []
        for line in lines:
            line = line.split("#")[
                0
            ].strip()  # Split at '#' and take the part before it
            if line:  # Only add non-empty lines
                result.extend(
                    line.split()
                )  # Split the line into elements and add them to the result
    return result


# Validates the instance data for consistency
def validateInstance():
    # Check if all activities can be completed within their time windows
    for activity in activities:
        if activity.duration > (activity.max_end - activity.min_start):
            raise ValueError(
                "Activity "
                + activity.id
                + " duration ("
                + (activity.duration / 3600)
                + "h) exceeds its time window"
            )

    # Check if workers' availability windows are valid
    for agent in agents:
        if agent.availability_start >= agent.availability_end:
            raise ValueError("Agent " + agent.id + " has invalid availability window")

    # Check if activities can be performed within workers' availability
    earliest_start = min([act.min_start for act in activities])
    latest_end = max([act.max_end for act in activities])

    for agent in agents:
        if (
            agent.availability_start > earliest_start
            or agent.availability_end < latest_end
        ):
            print(
                "Warning: Agent "
                + str(agent.id)
                + " availability might not cover all tasks"
            )


with hexaly.optimizer.HexalyOptimizer() as optimizer:
    #
    # Read path of input file
    #
    file_it = iter(read_elem(sys.argv[1]))

    # Read dimensions of the problem
    nb_agents = int(next(file_it))
    nb_activities = int(next(file_it))
    time_horizon = int(next(file_it))
    shift_increment = round(float(next(file_it)) * 60)

    # Read activities informations
    # Duration time of each activity
    activities = []
    for i in range(nb_activities):
        activities.append(Activity(i, 0, 0, int(next(file_it)) * 3600))
    # Maximum end and minimum possible start hour of each activity
    for i in range(nb_activities):
        activities[i].min_start = int(next(file_it)) * 3600
        activities[i].max_end = int(next(file_it)) * 3600

    # Read agents informations
    # Availability period in the day of each agent
    agents = [
        Agent(i, int(next(file_it)), int(next(file_it))) for i in range(nb_agents)
    ]

    # Generation of all possible shifts for each activity with a given increment
    shifts = generateShifts(activities, shift_increment)

    # Validates the instance data
    validateInstance()

    #
    # Declare the optimization model
    #
    model = optimizer.model

    # Decision variables : agent_shift[a][s] is true when the agent a works the shift s
    agent_shift = [[model.bool() for j in range(len(shifts))] for i in range(nb_agents)]

    # Constraints
    # 1. The shifts performed by an agent must not overlap
    for a in range(nb_agents):
        for s1 in shifts:
            for s2 in shifts:
                if incompatibleShifts(s1, s2) and s1 != s2:
                    model.constraint(agent_shift[a][s1.id] + agent_shift[a][s2.id] <= 1)

    # 2. An agent must take at least a one-hour break between two shifts
    for a in range(nb_agents):
        for s1 in shifts:
            for s2 in shifts:
                if notEnoughBreak(s1, s2) and s1 != s2:
                    model.constraint(agent_shift[a][s1.id] + agent_shift[a][s2.id] <= 1)

    # 3. Each agent must work exactly two shifts in the day
    for a in range(nb_agents):
        model.constraint(sum([agent_shift[a][s.id] for s in shifts]) == 2)

    # Objective 1 :  Minimize understaffing
    # For each activity and at each interval of time of the increment length
    total_understaffing = 0
    for activity in activities:
        current_time = activity.min_start
        while current_time < activity.max_end:
            # Current interval of time
            current_interval_start = current_time
            current_interval_end = current_time + shift_increment
            # The activity will not be pursued after its maximum possible end
            if current_interval_end > activity.max_end:
                current_interval_end = activity.max_end
            # Shifts performed in the current time interval
            shifts_at_current = []
            for shift in shifts:
                if shift.activity_id != activity.id:
                    continue
                if overlappingIntervals(
                    current_interval_start,
                    current_interval_end,
                    shift.shift_start,
                    shift.shift_end,
                ):
                    shifts_at_current.append(shift)
            # Number of agents currently performing the task
            total_agents_working = 0
            for shift in shifts_at_current:
                agents_working = sum(
                    [agent_shift[a][shift.id] for a in range(nb_agents)]
                )
                total_agents_working += agents_working
            # Number of agents missing
            total_understaffing += model.max(0, 1 - total_agents_working)
            current_time += shift_increment
    total_understaffing *= shift_increment
    model.minimize(total_understaffing)

    # Objective 2 : Minimize the total worktime of all the agents
    total_working_time = 0
    for a in range(nb_agents):
        total_working_time += sum(
            [agent_shift[a][s.id] * activities[s.activity_id].duration for s in shifts]
        )
    model.minimize(total_working_time)
    model.close()

    # Parameterize the optimizer
    if len(sys.argv) >= 4:
        optimizer.param.time_limit = int(sys.argv[3])
        optimizer.param.verbosity = 2
    else:
        optimizer.param.time_limit = 5
    optimizer.solve()
